<a href="https://colab.research.google.com/github/linoy25/Project-OnlyPlants/blob/main/Tutorials/tirgul5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from nltk.stem import PorterStemmer

def fetch_page(url):
    headers = {
        "User-Agent": "BraudeStudentProject/1.0 (student@braude.ac.il)"
    }

    try:
        response = requests.get(url, headers=headers)
        print("HTTP status:", response.status_code)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            return soup
        else:
            print("Response head:", response.text[:200])
            return None

    except Exception as e:
        print("Request error:", e)
        return None

def index_words(soup):
  index = {}
  words = re.findall(r'\w+', soup.get_text())
  for word in words:
    word = word.lower()
    if word in index:
      index[word] += 1
    else:
      index[word] = 1
  return index

def remove_stop_words(index):
  stop_words = {'a', 'an', 'the', 'and', 'or', 'in', 'on', 'at'}
  for stop_word in stop_words:
    if stop_word in index:
      del index[stop_word]
  return index

def apply_stemming(index):
  stemmer = PorterStemmer()
  stemmed_index = {}
  for word, count in index.items():
    stemmed_word = stemmer.stem(word)
    if stemmed_word in stemmed_index:
      stemmed_index[stemmed_word] += count
    else:
      stemmed_index[stemmed_word] = count
  return stemmed_index

def search(query, index):
  stemmer = PorterStemmer()
  query_words = re.findall(r'\w+', query.lower())
  results = {}
  for word in query_words:
    word = stemmer.stem(word)
    if word in index:
      results[word] = index[word]
    else:
      results[word] = 0 # מונע קריסה אם המילה לא מופיעה
  return results

def search_engine(url, query):
  soup = fetch_page(url)
  if soup is None:
     return None
  index = index_words(soup)
  index = remove_stop_words(index)
  index = apply_stemming(index)
  results = search(query, index)
  return results

# פתרון תרגיל 1: הרצה והשוואת דירוגים
url = 'https://en.wikipedia.org/wiki/ORT_Braude_College_of_Engineering'
queries = ["Industry", "Braude college", "Galilee center"]

# נשלוף את הדף פעם אחת כדי לא להעמיס
soup = fetch_page(url)
if soup:
    index = index_words(soup)
    index = remove_stop_words(index)
    index = apply_stemming(index)

    print("\n--- Original Ranking Method ---")
    for query in queries:
        results = search(query, index)
        rank = 1.0
        for word, count in results.items():
            if count > 0:
                rank = rank * (1 / count)
            else:
                rank = 1 # אם המילה לא נמצאה, נאפס את הדירוג כדי למנוע חלוקה באפס
        rank = 1 - rank
        print(f"Query: '{query}' | Results: {results} | Rank: {rank}")

    print("\n--- New Ranking Method (Sum of Frequencies) ---")
    for query in queries:
        results = search(query, index)
        # שיטת דירוג חדשה: פשוט לחבר את כמות הפעמים שכל מילה הופיעה
        new_rank = sum(results.values())
        print(f"Query: '{query}' | Results: {results} | Rank Score: {new_rank}")

# **# Concluding exercise**

In [ ]:
# 1. Install required libraries
!pip install firebase beautifulsoup4 requests

import requests
from bs4 import BeautifulSoup
import re
from firebase import firebase

# IMPORTANT: Replace this URL with your specific Firebase Realtime Database URL
DB_URL = 'https://plant-monitor-cloud-default-rtdb.europe-west1.firebasedatabase.app/'

# Initialize the Firebase connection
FBconn = firebase.FirebaseApplication(DB_URL, None)

# Define 10 significant words for the Orchid monitoring project
TARGET_WORDS = ['orchid', 'disease', 'sensor', 'temperature', 'moisture', 'light', 'root', 'leaf', 'health', 'monitor']

# Fetch the Wikipedia article about Orchids
# We use a custom User-Agent to prevent the server from blocking our request
url = 'https://en.wikipedia.org/wiki/Orchidaceae'
headers = {"User-Agent": "BraudeStudentProject/1.0 (student@braude.ac.il)"}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    print("Success: Website fetched and Firebase connection established!")
else:
    print(f"Error fetching the website. Status code: {response.status_code}")

In [ ]:
# Extract all words from the parsed HTML text and convert them to lowercase
all_words = re.findall(r'\w+', soup.get_text().lower())

# Initialize a dictionary (index) with our 10 target words, starting at count 0
orchid_index = {word: 0 for word in TARGET_WORDS}

# Count the occurrences of our target words in the article
for word in all_words:
    if word in orchid_index:
        orchid_index[word] += 1

print("Generated Index for the Orchid Project:", orchid_index)

# Save the complete index dictionary to the Firebase Realtime Database
# We use the 'put' method to store it cleanly under the key 'orchid_project_index' [cite: 753, 773]
FBconn.put('/', 'orchid_project_index', orchid_index)
print("Success: Index saved to Firebase Realtime Database!")

In [ ]:
import matplotlib.pyplot as plt

# Retrieve the saved index from the Firebase database [cite: 730, 750]
retrieved_data = FBconn.get('/orchid_project_index', None)

if retrieved_data:
    print("Success: Data retrieved from Firebase:")
    print(retrieved_data)

    # Prepare the data for plotting
    words_list = list(retrieved_data.keys())
    counts_list = list(retrieved_data.values())

    # Create the bar chart
    plt.figure(figsize=(10, 6))
    plt.bar(words_list, counts_list, color='darkseagreen')

    # Add title and labels to the chart
    plt.title('Word Frequencies - Orchid Project Index', fontsize=14, fontweight='bold')
    plt.xlabel('Project Concepts', fontsize=12)
    plt.ylabel('Appearance Count', fontsize=12)

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45)

    # Add a horizontal grid for easier value reading
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    # Display the chart
    plt.tight_layout() # Ensures labels don't get cut off
    plt.show()
else:
    print("Error: No data found at the specified Firebase path.")
    print("Make sure Block 2 ran successfully and your database rules allow read access (.read: true).")